# Day 12 Tutorial：训练内调参，不偷看测试集

## Goal

只用训练区域内部 5 折，从预先声明的 Ridge `alpha` 候选中选择参数；冻结后在外部验证区评价一次。教程不创建测试集。


## Setup

人工数据固定生成并分成训练/外部验证。填补和标准化都位于被搜索的 Pipeline 内。


In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_SEED = 42
SPLIT_SEED = 17
CV_SEED = 42
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
print({"python": platform.python_version(), "sklearn": sklearn.__version__, "alphas": ALPHAS})


## Steps

### 1. 固定训练区与外部验证区


In [ ]:
X, y = make_regression(
    n_samples=110, n_features=8, n_informative=6, noise=20.0, random_state=DATA_SEED
)
# 固定位置加入少量缺失值，迫使填补器成为流程的一部分。
X[[2, 19, 37, 66, 93], [0, 3, 5, 1, 6]] = np.nan
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SPLIT_SEED
)
print({"train": X_train.shape, "external_validation": X_valid.shape})


### 2. 在训练内部执行无泄漏网格搜索


In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge()),
])
param_grid = {"ridge__alpha": ALPHAS}
cv = KFold(n_splits=5, shuffle=True, random_state=CV_SEED)
search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    return_train_score=True,
    refit=True,
)
search.fit(X_train, y_train)
print("Frozen parameters selected inside training only:", search.best_params_)


### 3. 保留完整候选表并恢复负评分


In [ ]:
raw_results = pd.DataFrame(search.cv_results_)
candidate_table = raw_results[[
    "param_ridge__alpha", "mean_train_score", "mean_test_score",
    "std_test_score", "rank_test_score"
]].copy()
candidate_table = candidate_table.rename(columns={"param_ridge__alpha": "alpha"})
candidate_table["mean_train_rmse"] = -candidate_table["mean_train_score"]
candidate_table["mean_cv_rmse"] = -candidate_table["mean_test_score"]
candidate_table["std_cv_rmse"] = candidate_table["std_test_score"]
candidate_table["cv_minus_train_rmse"] = (
    candidate_table["mean_cv_rmse"] - candidate_table["mean_train_rmse"]
)
candidate_table = candidate_table.sort_values("alpha").reset_index(drop=True)
display(candidate_table[[
    "alpha", "mean_train_rmse", "mean_cv_rmse", "std_cv_rmse",
    "cv_minus_train_rmse", "rank_test_score"
]].round(4))


### 4. 参数冻结后外部验证一次


In [ ]:
external_valid_prediction = search.best_estimator_.predict(X_valid)
external_valid_rmse = float(root_mean_squared_error(y_valid, external_valid_prediction))
external_result = pd.DataFrame([{
    "split": "external_validation",
    "frozen_alpha": search.best_params_["ridge__alpha"],
    "rmse": external_valid_rmse,
}])
display(external_result.round(4))

ax = candidate_table.plot(
    x="alpha", y="mean_cv_rmse", yerr="std_cv_rmse", marker="o", capsize=4, legend=False
)
ax.set_xscale("log")
ax.set(title="Day 12 training-internal CV (synthetic data)", ylabel="mean CV RMSE")
plt.tight_layout()
plt.show()


## Checks

确认候选完整、最佳参数来自候选表、外部结果有限，并且 notebook 没有创建测试数组。


In [ ]:
assert len(candidate_table) == len(ALPHAS)
assert set(candidate_table["alpha"].astype(float)) == set(ALPHAS)
assert search.best_params_["ridge__alpha"] in ALPHAS
assert np.isfinite(external_valid_rmse)
assert X_train.shape[1] == X_valid.shape[1]
assert "X_test" not in globals() and "y_test" not in globals()

print("Checks passed: search used training data; external validation was evaluated after freezing.")


## Next Steps

在本人实验中保存完整 `cv_results_` 和预注册候选，不只保存最佳一行。如果多轮依据同一外部验证集修改候选，要如实把它视为开发反馈；无偏地同时调参与估计性能需要嵌套交叉验证或新的独立评价。
